<a href="https://colab.research.google.com/github/Rajeev-Shyam/Quellgeist/blob/main/Quell_Capability_%2B_Gemini_reference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
# Reduces CUDA fragmentation — must be set before torch initialises CUDA.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
%pip install -q --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo
%pip install -q --upgrade google-genai
# If a Qwen3.5 load later raises "model class not found": re-run this cell, then
# Runtime ▸ Restart runtime. Import unsloth BEFORE transformers if it nags.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 36.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 322.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 376.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 293.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 244.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 278.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 231.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 MB 245.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 302.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 301.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 201.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 212.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import unsloth          # import first per Unsloth's patching recommendation
import torch, transformers
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
print("transformers:", transformers.__version__, "| unsloth:", unsloth.__version__)
# Expected: CUDA True, Tesla T4, ~15-16 GB, transformers 5.x

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
CUDA: True Tesla T4
VRAM GB: 15.6
transformers: 5.5.0 | unsloth: 2026.6.7


In [ ]:
import json

FIXTURE = {
  "id": "bad_deploy_0001",
  "failure_class": "bad_deploy",
  "now": "2026-06-18T10:15:00Z",
  "logs": [
    {"ts": "2026-06-18T09:55:01Z", "level": "INFO",  "route": "/login", "status": 200, "msg": "login ok"},
    {"ts": "2026-06-18T10:01:30Z", "level": "INFO",  "route": "/login", "status": 200, "msg": "login ok"},
    {"ts": "2026-06-18T10:02:12Z", "level": "ERROR", "route": "/login", "status": 500, "msg": "TypeError: 'NoneType' object is not subscriptable in auth.verify_token"},
    {"ts": "2026-06-18T10:03:05Z", "level": "ERROR", "route": "/login", "status": 500, "msg": "TypeError: 'NoneType' object is not subscriptable in auth.verify_token"},
    {"ts": "2026-06-18T10:07:44Z", "level": "ERROR", "route": "/login", "status": 500, "msg": "TypeError: 'NoneType' object is not subscriptable in auth.verify_token"},
    {"ts": "2026-06-18T10:08:10Z", "level": "INFO",  "route": "/data",  "status": 200, "msg": "data served"}
  ],
  "commits": [
    {"sha": "a1b2c3d", "ts": "2026-06-18T10:01:50Z", "msg": "deploy: refactor token parsing", "files": ["src/app/auth.py"]},
    {"sha": "9f8e7d6", "ts": "2026-06-17T16:20:00Z", "msg": "docs: update README", "files": ["README.md"]}
  ],
  "gold_cause": "Bad deploy a1b2c3d (10:01:50Z) refactored auth.py and introduced a NoneType error in verify_token; /login 500s begin ~20s later at 10:02:12Z.",
  "gold_evidence": ["log:2026-06-18T10:02:12Z /login 500", "commit:a1b2c3d src/app/auth.py"]
}
with open("bad_deploy_0001.json", "w") as f:
    json.dump(FIXTURE, f, indent=2)   # later: evals/scenarios/fixtures/bad_deploy_0001.json

SYSTEM = """You are an incident-triage agent. Diagnose the production incident using ONLY evidence
returned by tools. On each turn, output EXACTLY ONE JSON object and nothing else.

To call a tool:
  {"action": "query_logs", "args": {"level": "ERROR"}}
  {"action": "get_recent_commits", "args": {"n": 5}}

When you have enough evidence, output your final answer:
  {"action": "final",
   "insufficient_evidence": false,
   "hypotheses": [
     {"cause": "<one sentence>", "confidence": 0.0-1.0,
      "evidence": ["log:<ts> <route> <status>", "commit:<sha> <file>"]}
   ]}

Rules: rank hypotheses most-likely first. Every evidence string MUST correspond to a real row a
tool returned. If the tools do not support a cause, set "insufficient_evidence": true and do not invent one.
"""

In [ ]:
# Cell S4 — VLM-aware chat rendering (works for text tokenizers AND VLM processors)
def _as_parts(messages):
    out = []
    for m in messages:
        c = m["content"]
        out.append(m if not isinstance(c, str)
                   else {"role": m["role"], "content": [{"type": "text", "text": c}]})
    return out

def render_chat(tok, messages, tokenize=True, add_generation_prompt=True):
    """Try string content first (Qwen3-4B path); fall back to multimodal parts (Qwen3.5 VLM)."""
    last = None
    for msgs in (messages, _as_parts(messages)):
        for extra in ({"enable_thinking": False}, {}):   # prefer thinking-off; drop kwarg if unsupported
            try:
                return tok.apply_chat_template(
                    msgs, tokenize=tokenize, add_generation_prompt=add_generation_prompt,
                    return_tensors=("pt" if tokenize else None), **extra)
            except (TypeError, KeyError) as e:
                last = e
    raise last

Session 1: A1-A4

In [ ]:
import re, json

def query_logs(level=None, since=None):
    rows = FIXTURE["logs"]
    if level: rows = [r for r in rows if r["level"] == level]
    if since: rows = [r for r in rows if r["ts"] >= since]
    return rows

def get_recent_commits(n=5):
    return FIXTURE["commits"][:n]

TOOLS = {"query_logs": query_logs, "get_recent_commits": get_recent_commits}

def extract_json(text):
    if not text:
        return None
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)   # strip reasoning
    text = re.sub(r"```(?:json)?|```", "", text)                      # strip markdown fences
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)                   # greedy: first { … last }
    if m:
        try:
            return json.loads(m.group(0))
        except json.JSONDecodeError:
            pass
    start = text.find("{")                                            # fallback: first balanced object
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start:i + 1])
                except json.JSONDecodeError:
                    return None
    return None

def _tk(tok):
    return getattr(tok, "tokenizer", tok)   # processor -> inner tokenizer; tokenizer -> itself

def chat(model, tok, messages, max_new_tokens=512):
    tk = _tk(tok)
    inputs = render_chat(tok, messages).to(model.device)
    out = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tk.eos_token_id)
    return tk.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)


def run_loop(model, tok, max_steps=4, verbose=True):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": f"Incident at {FIXTURE['now']}. /login is failing. Diagnose."}]
    trace = []
    for step in range(max_steps):
        raw = chat(model, tok, messages)
        obj = extract_json(raw)
        if verbose: print(f"--- step {step} ---\n{obj}\n")
        if obj is None:
            trace.append(("parse_error", raw)); break
        if obj.get("action") == "final":
            trace.append(("final", obj)); return obj, trace
        tool = TOOLS.get(obj.get("action"))
        if not tool:
            trace.append(("bad_tool", obj)); break
        result = tool(**obj.get("args", {}))
        trace.append((obj["action"], result))
        messages.append({"role": "assistant", "content": json.dumps(obj)})
        messages.append({"role": "user", "content": f"TOOL RESULT:\n{json.dumps(result)}"})
    return None, trace

def free():
    import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
import time
from unsloth import FastModel     # VERIFY: FastModel + these kwargs postdate my cutoff
model35, tok35 = FastModel.from_pretrained(
    model_name="unsloth/Qwen3.5-4B",
    max_seq_length=4096,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
)
FastModel.for_inference(model35)
# If this OOMs on a free T4: drop max_seq_length to 2048, or load_in_4bit=True for
# inference only — and RECORD that you had to.

t0 = time.time()
final35, trace35 = run_loop(model35, tok35)
print("FINAL:", json.dumps(final35, indent=2))
print(f"steps={len([t for t in trace35 if t[0] in TOOLS or t[0]=='final'])}  wall={time.time()-t0:.1f}s")
# RECORD: (a) called a tool? (b) used the result? (c) deploy ranked cause #1 w/ valid evidence?
#         (d) no fabricated evidence? + step count + wall-clock.
del model35, tok35; free()

==((====))==  Unsloth 2026.6.7: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/904 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/876 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

--- step 0 ---
{'action': 'query_logs', 'args': {'level': 'ERROR'}}

--- step 1 ---
{'action': 'get_recent_commits', 'args': {'n': 5}}

--- step 2 ---
{'action': 'final', 'insufficient_evidence': False, 'hypotheses': [{'cause': 'A recent commit to auth.py introduced a bug where auth.verify_token receives None instead of a valid token object, causing a TypeError during login.', 'confidence': 0.95, 'evidence': ["log:2026-06-18T10:02:12Z /login 500 TypeError: 'NoneType' object is not subscriptable in auth.verify_token", 'commit:a1b2c3d 2026-06-18T10:01:50Z src/app/auth.py refactor token parsing']}]}

FINAL: {
  "action": "final",
  "insufficient_evidence": false,
  "hypotheses": [
    {
      "cause": "A recent commit to auth.py introduced a bug where auth.verify_token receives None instead of a valid token object, causing a TypeError during login.",
      "confidence": 0.95,
      "evidence": [
        "log:2026-06-18T10:02:12Z /login 500 TypeError: 'NoneType' object is not subscriptable

In [ ]:
import time, json
t0 = time.time()
final35, trace35 = run_loop(model35, tok35)
print("FINAL:", json.dumps(final35, indent=2))
print(f"wall={time.time()-t0:.1f}s")

NameError: name 'model35' is not defined

In [ ]:
import time
from unsloth import FastLanguageModel
model3, tok3 = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B",
    max_seq_length=4096,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model3)

t0 = time.time()
final3, trace3 = run_loop(model3, tok3)
print("FINAL:", json.dumps(final3, indent=2))
print(f"wall={time.time()-t0:.1f}s")
# RECORD the same four criteria + step count + wall-clock.
#del model3, tok3; free()

==((====))==  Unsloth 2026.6.7: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.70k [00:00<?, ?B/s]

...

unsloth/qwen3-4b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

--- step 0 ---
{'action': 'query_logs', 'args': {'level': 'ERROR'}}



Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- step 1 ---
{'action': 'get_recent_commits', 'args': {'n': 5}}

--- step 2 ---
{'action': 'final', 'insufficient_evidence': False, 'hypotheses': [{'cause': 'The /login route is failing due to a TypeError in auth.verify_token when trying to subscript a NoneType object', 'confidence': 0.95, 'evidence': ["log:2026-06-18T10:02:12Z /login 500 TypeError: 'NoneType' object is not subscriptable in auth.verify_token", "log:2026-06-18T10:03:05Z /login 500 TypeError: 'NoneType' object is not subscriptable in auth.verify_token", "log:2026-06-18T10:07:44Z /login 500 TypeError: 'NoneType' object is not subscriptable in auth.verify_token", 'commit:a1b2c3d deploy: refactor token parsing src/app/auth.py']}]}

FINAL: {
  "action": "final",
  "insufficient_evidence": false,
  "hypotheses": [
    {
      "cause": "The /login route is failing due to a TypeError in auth.verify_token when trying to subscript a NoneType object",
      "confidence": 0.95,
      "evidence": [
        "log:2026-06-18T10:02:12

In [ ]:
from google import genai                          # newer SDK; verify it's current
try:
    from google.colab import userdata
    def get_secret(k): return userdata.get(k)
except ImportError:
    from kaggle_secrets import UserSecretsClient   # Kaggle path
    def get_secret(k): return UserSecretsClient().get_secret(k)

GEMINI_MODEL = "gemini-2.5-flash"   # VERIFY current free-tier id at ai.google.dev
client = genai.Client(api_key=get_secret("GEMINI_API_KEY"))

ref_prompt = (SYSTEM
    + "\n\nLOGS:\n" + json.dumps(FIXTURE["logs"])
    + "\n\nCOMMITS:\n" + json.dumps(FIXTURE["commits"])
    + "\n\nReturn only the final JSON object.")
resp = client.models.generate_content(model=GEMINI_MODEL, contents=ref_prompt)
print(extract_json(resp.text))
# RECORD as the quality bar the local 4B is measured against.
# --- Legacy SDK fallback if `google-genai` is unavailable: ---
# import google.generativeai as genai
# genai.configure(api_key=get_secret("GEMINI_API_KEY"))
# resp = genai.GenerativeModel(GEMINI_MODEL).generate_content(ref_prompt)
# print(extract_json(resp.text))

{'action': 'final', 'insufficient_evidence': False, 'hypotheses': [{'cause': 'A recent deployment with refactored token parsing in the authentication module introduced a bug causing a TypeError when verifying tokens.', 'confidence': 1.0, 'evidence': ['commit:a1b2c3d 2026-06-18T10:01:50Z deploy: refactor token parsing', 'log:2026-06-18T10:02:12Z /login 500', "log:2026-06-18T10:02:12Z ERROR TypeError: 'NoneType' object is not subscriptable in auth.verify_token"]}]}


In [ ]:
print(repr(resp.text)[:500])

'```json\n{\n  "action": "final",\n  "insufficient_evidence": false,\n  "hypotheses": [\n    {\n      "cause": "A recent deployment introduced a bug in the token parsing logic, leading to \'NoneType\' errors during login verification.",\n      "confidence": 1.0,\n      "evidence": [\n        "log:2026-06-18T10:02:12Z /login 500",\n        "log:2026-06-18T10:02:12Z ERROR TypeError: \'NoneType\' object is not subscriptable in auth.verify_token",\n        "commit:a1b2c3d 2026-06-18T10:01:50Z",\n   


In [ ]:
print(extract_json(resp.text))

{'action': 'final', 'insufficient_evidence': False, 'hypotheses': [{'cause': 'A recent deployment with refactored token parsing in the authentication module introduced a bug causing a TypeError when verifying tokens.', 'confidence': 1.0, 'evidence': ['commit:a1b2c3d 2026-06-18T10:01:50Z deploy: refactor token parsing', 'log:2026-06-18T10:02:12Z /login 500', "log:2026-06-18T10:02:12Z ERROR TypeError: 'NoneType' object is not subscriptable in auth.verify_token"]}]}


Session 2: B1-B2

In [ ]:
from datasets import Dataset

def to_text(tok, logs, commits, gold):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"LOGS:\n{json.dumps(logs)}\nCOMMITS:\n{json.dumps(commits)}"},
            {"role": "assistant", "content": json.dumps(gold)}]
    return render_chat(tok, msgs, tokenize=False, add_generation_prompt=False)

gold_answer = {"action": "final", "insufficient_evidence": False,
    "hypotheses": [{"cause": FIXTURE["gold_cause"], "confidence": 0.9, "evidence": FIXTURE["gold_evidence"]}]}
rows = [{"logs": FIXTURE["logs"], "commits": FIXTURE["commits"], "gold": gold_answer} for _ in range(12)]
# 12 identical rows is fine for a SMOKE test (does it train/loss-drop/export). Real,
# varied data comes in Wave 4 — and must be a different distribution from the holdout (DR-0003).

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

m, t = FastLanguageModel.from_pretrained("unsloth/Qwen3-4B", max_seq_length=2048, load_in_4bit=True)
m = FastLanguageModel.get_peft_model(
    m, r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407)

ds = Dataset.from_list([{"text": to_text(t, r["logs"], r["commits"], r["gold"])} for r in rows])
trainer = SFTTrainer(
    model=m, tokenizer=t, train_dataset=ds,   # if TypeError on `tokenizer=`, use processing_class=t
    args=SFTConfig(max_seq_length=2048, dataset_text_field="text",
        per_device_train_batch_size=1, gradient_accumulation_steps=4,
        warmup_steps=2, max_steps=30, learning_rate=2e-4,
        logging_steps=1, output_dir="ft_qwen3_4b", report_to="none"))
print(trainer.train())                            # RECORD: loss trend + seconds/step
m.save_pretrained_gguf("ft_qwen3_4b_gguf", t)     # RECORD: export succeeds (expected OK — Qwen3 dense)

==((====))==  Unsloth 2026.6.7: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

unsloth/qwen3-4b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.6.7 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/12 [00:00<?, ? examples/s]

...


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [09:35<00:00, 287.76s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [04:21<00:00, 130.53s/it]


Unsloth: Merge process complete. Saved to `/content/ft_qwen3_4b_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b9703 (llama-b9703-bin-ubuntu-x64.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['ft_qwen3_4b_gguf_gguf/qwen3-4b.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q8_0. This might take 10 minutes...


KeyboardInterrupt: 

Session 3: C1-C2

In [ ]:
from unsloth import FastModel
from trl import SFTTrainer, SFTConfig

m, t = FastModel.from_pretrained("unsloth/Qwen3.5-4B", max_seq_length=2048,
        load_in_4bit=False, load_in_16bit=True, full_finetuning=False)
m = FastModel.get_peft_model(
    m, r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407)

ds = Dataset.from_list([{"text": to_text(t, r["logs"], r["commits"], r["gold"])} for r in rows])
trainer = SFTTrainer(
    model=m, tokenizer=t, train_dataset=ds,   # if TypeError on `tokenizer=`, use processing_class=t
    args=SFTConfig(max_seq_length=2048, dataset_text_field="text",
        per_device_train_batch_size=1, gradient_accumulation_steps=4,
        warmup_steps=2, max_steps=30, learning_rate=2e-4,
        logging_steps=1, output_dir="ft_qwen35_4b", report_to="none"))
print(trainer.train())   # RECORD: fit in VRAM? loss trend? seconds/step (expect SLOWER — GDN kernels)
# If this OOMs on a cold T4: reduce max_seq_length to 1024. If it still won't fit/train at
# acceptable speed, RECORD "Qwen3.5-4B not free-T4-trainable as configured" — a valid finding.

In [ ]:
m.save_pretrained_gguf("ft_qwen35_4b_gguf", t)
# RECORD: did export succeed? If it fails on the GDN/MoE architecture, that is a FINDING,
# not a blocker — no clean GGUF means awkward Ollama serving, which favours the Qwen3-4B default.